# RAG를 위한 PREPROCESS2

## Splitter

1. 토큰 제한 회피: 대부분의 LLM은 입력으로 받을 수 있는 **최대 토큰 수(컨텍스트 창)**에 제한이 있습니다. 긴 문서를 통째로 넣으려 하면 오류가 발생하거나, 문서의 중요한 부분이 잘릴 수 있습니다.
2. 임베딩 효율성: 텍스트를 벡터 저장소에 임베딩할 때, 너무 큰 청크는 특정 질문에 대한 정확한 의미 정보를 희석시킬 수 있습니다. 적절한 크기로 분할해야 검색 정확도가 높아집니다.
3. 처리 속도 및 비용: 작은 청크를 사용하면 LLM이 처리할 데이터의 양이 줄어들어 속도가 빨라지고 비용이 절감됩니다.

**텍스트 분할기(Text Splitter)**는 방대한 텍스트 문서나 데이터를 관리하고 처리하기 위한 핵심 도구이다. 이 객체의 주된 목적은 긴 텍스트를 **더 작고 관리하기 쉬운 청크(Chunk)**로 나누는 것

(토큰이랑 다르다. 청크는 RAG, 검색, 요약 같은 작업을 위해 입력 텍스트를 나누는 단위이다. 청크는 덩어리임 토큰은 모델이 텍스트를 읽는 최소단위고!)

> 결국 쪼개는 것이다. 긴 문서를 통째로 넣으면 문제가 있다. **LLM이 감당할 수 있는 컨텍스트 length보다 많으면 동작하지 않기 때문에 문서를 쪼개는 것이다.**  (max toekn error 방지 위해 문서를 청크로 나눈다)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter, MarkdownTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# chunk_size: 생성될 각 청크의 최대 크기를 정의합니다. (일반적으로 문자 수 또는 토큰 수 기준)
# chunk_overlap: 연속된 청크 간에 겹치는 텍스트의 양을 정의합니다.
# 목적: 청크 경계에서 중요한 의미나 문맥이 단절되는 것을 방지하기 위해 사용됩니다. 적절한 오버랩을 설정하면, 한 청크의 끝과 다음 청크의 시작 부분이 문맥을 이어갈 수 있습니다.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

FILE_PATH = "./SPRi AI Brief_6월호_산업동향_F.pdf"
loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

chunks = text_splitter.split_documents(documents)
print(f"총 {len(chunks)}개 청크 생성")
print(chunks[0].page_content[:200])
print(chunks[0].metadata) 


  chunk_size=250

  한 조각의 최대 길이입니다.
  여기서는 한 chunk를 대략 250자 이하로 자르겠다는 뜻입니다.

  일반 기준:

  - 짧은 질의응답: 300~700
  - 일반 문서 RAG: 500~1000
  - 긴 문맥이 필요한 문서: 1000~2000
  - 너무 작으면 문맥이 끊기고, 너무 크면 검색 정확도가 떨어질 수 있음

  ———

  chunk_overlap=50

  chunk끼리 겹치게 할 문자 수입니다.
  앞 chunk의 끝부분 50자를 다음 chunk에도 포함합니다.

  이유는 문장이 chunk 경계에서 잘려도 문맥이 이어지게 하기 위해서입니다.

  일반 기준:

  - 보통 chunk_size의 10~20%
  - 예: chunk_size=500이면 overlap=50~100
  - 너무 크면 중복 저장이 많아지고 비용이 늘어남

  ———

  length_function=len

  길이를 어떻게 계산할지 정합니다.

  len

  은 파이썬 기본 길이 계산 함수입니다.
  즉, 문자열의 문자 수를 기준으로 chunk 크기를 계산합니다.

  일반 기준:

  - 간단히 쓸 때는 len
  - 모델 토큰 수 기준으로 정확히 자르고 싶으면 토큰 기반 length function 사용

  ———

  is_separator_regex=False

  구분자를 정규표현식으로 해석할지 여부입니다.

  False이면 구분자를 일반 문자열로 봅니다.
  보통 기본값처럼 False를 많이 씁니다.

  예:

  "\n\n"

  을 진짜 줄바꿈 문자열로 봅니다.

  True이면 정규표현식 패턴으로 해석합니다.

  ———

In [ ]:
def create_production_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=1200,           # BGE, text-embedding-3-large 최적
        chunk_overlap=250,         # 문맥 연결 완벽
        length_function=len,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
        add_start_index=True,
        strip_whitespace=True,
    )

# 사용법
splitter = create_production_splitter()
chunks = splitter.split_documents(documents)

# 메타데이터 확인 (출처 인용용)
for i, chunk in enumerate(chunks[:3]):
    print(f"청크 {i+1}: {len(chunk.page_content)}자")
    print(f"출처: {chunk.metadata.get('source')} 페이지: {chunk.metadata.get('page')}")
    print(f"시작 위치: {chunk.metadata.get('start_index')}")
    print("-" * 50)

예시 결과:

청크 1: 22자  
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 0  
시작 위치: 0  

청크 2: 878자  
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 1  
시작 위치: 0  

청크 3: 19자  
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 2   
시작 위치: 0  



## VectorStore

Chorma DB 또는 Fassis를 사용해서 벡터 스토어를 만들 수 있다! (크로마와 파이쓰는 랭체인과 연결이 쉽다)

In [ ]:
pip install langchain langchain-google-genai langchain-community 
pip install chromadb faiss-cpu pymilvus 

## ChromaDB

Chroma는 파이썬 패키지로 쉽게 설치하고 사용할 수 있으며, 로컬에서 실행하거나 서버로 배포할 수 있는 완전한 기능을 갖춘 벡터 데이터베이스입니다. LangChain과의 연동이 매우 쉽습니다.

🛠️ 특징
설치 용이: Python pip 설치 후 바로 사용 가능합니다.

유연성: 영구 저장소(로컬 파일 시스템) 또는 인메모리(RAM) 모드를 모두 지원합니다.

메타데이터 필터링: 메타데이터를 기반으로 검색 결과를 필터링하는 기능이 강력합니다.

In [ ]:
import os
import time
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS

# Chroma 공식 Gemini 예제는 gemini-embedding-001 모델을 사용합니다.
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT"

In [ ]:
# Chroma 로컬 DB가 저장될 폴더를 명시적으로 준비합니다.
CHROMA_DB_DIR = os.path.abspath("./chroma_db_gemini")
os.makedirs(CHROMA_DB_DIR, exist_ok=True)

FILE_PATH = "./data/SPRi AI Brief_6월호_산업동향_F.pdf"
loader = PyPDFLoader(FILE_PATH)
docs = loader.load_and_split(
    RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)

In [ ]:
# 크로마에 들어가는 인자들이다. 
# - Documents (List[Document]): 벡터 저장소에 추가할 문서 리스트 
# - embedding (Optional[Embeddings]): 임베딩 함수. 기본값은 None
# - ids (Optional[List[str]]): 문서 ID 리스트. 기본값은 None
# - collection_name (str): 생성할 컬렉션 이름.
# - persist_directory (Optional[str]): 컬렉션을 저장할 디렉토리. 기본값은 None
# - client_settings (Optional[chromadb.config.Settings]): Chroma 클라이언트 설정
# - client (Optional[chromadb.Client]): Chroma 클라이언트 인스턴스
# - collection_metadata (Optional[Dict]): 컬렉션 구성 정보. 기본값은 None

# 1. 색인 & 저장
# persist_directory를 지정하면 데이터가 로컬에 영구적으로 저장됩니다.
# from_documents()는 Document의 page_content와 metadata를 함께 Chroma에 저장합니다.
# Gemini 무료 티어 임베딩 요청 제한을 피하기 위해 문서를 나누어 저장합니다.
batch_size = 50
sleep_seconds = 60

chromadb = Chroma.from_documents(
    documents=docs[:batch_size],
    embedding=embeddings,
    collection_name="AI_industry_2025_06",
    persist_directory=CHROMA_DB_DIR
)

print(f"1차 저장 완료: {chromadb._collection.count()}개 문서")

for start in range(batch_size, len(docs), batch_size):
    end = start + batch_size
    batch_docs = docs[start:end]

    print(f"{start}~{min(end, len(docs))}번 문서 추가 전 {sleep_seconds}초 대기")
    time.sleep(sleep_seconds)

    chromadb.add_documents(batch_docs)
    print(f"현재 저장 문서 수: {chromadb._collection.count()}개")

print(f"Chroma 저장 완료: {chromadb._collection.count()}개 문서")



In [ ]:
# 2. 검색
results = chromadb.similarity_search("AI 관련 내용", k=5) # 질문에 따라 문서들 중 몇개를 추출할 것이냐? 즉, 유사한거 5개 정도 뽑을래 이런 것임
for r in results:
    print(f"[페이지 {r.metadata['page']}] {r.page_content[:200]}...")

In [ ]:
collection = chromadb._collection # 크로마 DB 내부의 실제 collection 꺼내기! 이 collection으로 아래의 작업을 할 수 있다. 

all_data = collection.get(
    # 검색할 ID 리스트 ([] 또는 None이면 모든 ID를 검색)
    ids=None, 
    # 반환할 요소 지정 (ID, 문서 내용, 메타데이터 등)
    include=['metadatas', 'documents'] # IDs, 문서 내용, 메타데이터를 요청
)

print("--- 📝 색인된 문서 ID 목록 ---")
# 결과는 딕셔너리 형태로 반환되며, 'ids' 키에 모든 ID가 리스트로 들어 있습니다.
print(all_data['ids'])
print(all_data['metadatas'])

# 이 코드를 통해 DB의 모든 id와 메타데이터를 조회할 수 있다. 
# 물론 실제 실행은 collection 이 아니라, chromadb로 직접 한다. 

In [ ]:
# 3. ID로 정확히 삭제 
ids_to_delete = ["a589ed3c-796c-40e5-b2cb-c25384e24277"]
print(f"--- ⚠️ {len(ids_to_delete)}개의 문서를 제거합니다. ---")
chromadb.delete(ids=ids_to_delete)
print(f"삭제 후 문서 수: {chromadb._collection.count()}")

In [ ]:
# 4. 컬렉션 전체 삭제 
chromadb.delete_collection()
print("Chroma 컬렉션 전체 삭제 완료")

metadata 기반으로 문서 제거하기 가능함!

In [ ]:
# 1. 제거 조건 정의
# 예: 메타데이터에서 'source'가 'legacy_data.txt'인 모든 문서를 제거하고 싶을 때
# where 인수를 사용할 때는 반드시 $eq(equal), $gt(greater than) 등 ChromaDB의 연산자를 포함하여 조건을 명확하게 지정해야 합니다.
filter_condition = {
    "source": {
        "$eq": "legacy_data.txt" # 'source' 메타데이터 값이 정확히 'legacy_data.txt'인 경우
    }
}

print(f"--- ⚠️ 메타데이터 조건에 맞는 모든 문서를 제거합니다. ---")

# 2. where 인수를 사용하여 필터링 조건에 맞는 문서들을 삭제합니다.
chromadb.delete(
    where=filter_condition
)

print("--- ✅ 조건부 문서 제거 완료 ---")

DB에 문서 추가하기

In [ ]:
import os
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT"
)

CHROMA_DB_DIR = os.path.abspath("./chroma_db_gemini")
os.makedirs(CHROMA_DB_DIR, exist_ok=True)

db = Chroma(
    collection_name="AI_industry_2025_06",
    persist_directory=CHROMA_DB_DIR,
    embedding_function=embeddings
)

# add_documents 메서드를 사용하여 Document 객체 추가 (권장)
new_documents = [
    Document(
        page_content="LLM은 거대 언어 모델의 약자로, 방대한 데이터로 훈련됩니다.",
        metadata={"source": "new_article_1.txt", "date": "2025-10-01"}
    ),
    Document(
        page_content="벡터 저장소는 임베딩 벡터와 메타데이터를 저장하는 데이터베이스입니다.",
        metadata={"source": "new_article_2.txt", "date": "2025-10-02"}
    )
]

# 이 메서드는 내부적으로 텍스트를 임베딩하고 DB에 저장합니다.
added_ids = db.add_documents(new_documents)

print(f"--- ✅ 새로운 Document 객체 추가 완료 ---")
print(f"새로 추가된 항목의 ID: {added_ids}")


# 예상 출력: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
--- ✅ 새로운 Document 객체 추가 완료 ---
새로 추가된 항목의 ID: ['85524632-f958-405d-9262-cc0eb1443e82', '24832eee-3e5b-4aaa-85b9-d8aa93718f21']

In [ ]:
# 텍스트 문자열의 리스트만 가지고 있고, 각 텍스트에 동일한 메타데이터를 일괄 적용하고 싶거나 메타데이터가 필요 없을 때 사용합니다.
# 1. 사용할 텍스트 리스트
texts_to_add = [
    "인공지능은 딥러닝 기술의 발전으로 큰 진보를 이루었습니다.",
    "Python의 Requests 라이브러리는 웹 요청을 처리하는 데 유용합니다."
]

# 2. 모든 텍스트에 적용할 메타데이터 (선택 사항)
# 각 텍스트에 공통적으로 적용됩니다.
common_metadata = {"source": "manual_input", "type": "quick_note"}

# 3. add_texts() 메서드를 사용하여 추가
# texts_to_add의 각 문자열이 하나의 Document의 page_content가 됩니다.
added_ids_from_text = db.add_texts(
    texts=texts_to_add,
    metadatas=[common_metadata] * len(texts_to_add) # 메타데이터는 리스트로 제공해야 함
)

print(f"\n--- ✅ 텍스트 리스트 추가 완료 ---")
print(f"새로 추가된 항목의 ID: {added_ids_from_text}")


## Faiss

FAISS는 Facebook AI가 개발한 고성능 유사도 검색 라이브러리입니다. 특히 수백만, 수십억 개의 벡터를 처리할 때 매우 빠르지만, 순수한 인덱싱 라이브러리이므로 벡터만 저장하고 메타데이터 처리가 비교적 단순합니다.

🛠️ 특징
속도: 대규모 벡터 검색에서 최고 수준의 검색 속도를 자랑합니다.

인덱스: 벡터 저장에 특화된 다양한 인덱스 구조를 제공합니다.

경량: 별도의 데이터베이스 서버가 필요 없는 C++ 기반의 라이브러리입니다.

In [ ]:
# pip install faiss-cpu
# pip install langchain-community langchain-google-genai

In [ ]:
# 기본적으로 인-메모리(in-memory) 방식으로 작동하기 때문에, 데이터를 영구적으로 저장하고 다시 로드하는 과정이 필수적
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# 1. 임베딩 함수 정의
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# 2. 초기 Document 리스트
documents = [
    Document(
        page_content="Faiss는 벡터 유사도 검색을 위한 라이브러리입니다.",
        metadata={"source": "faq"}
    ),
    Document(
        page_content="이 라이브러리는 주로 C++로 작성되었으며 Python 바인딩을 제공합니다.",
        metadata={"source": "manual"}
    ),
    Document(
        page_content="Faiss는 대규모 데이터셋에 대한 효율적인 검색을 목표로 합니다.",
        metadata={"source": "faq"}
    )
]

# 3. FAISS 인덱스 생성 (메모리 로드)
# 여기서 모든 텍스트가 임베딩되고 Faiss 인덱스가 메모리에 생성됩니다.
vectorstore = FAISS.from_documents(documents, embeddings)

print("✅ Faiss 인덱스 생성 완료 및 메모리 로드됨")


In [ ]:
# 인덱스를 저장할 로컬 디렉토리 경로
local_path = "./faiss_index_store"

# vectorstore 객체를 지정된 경로에 저장합니다.
vectorstore.save_local(local_path)

print(f"✅ Faiss 인덱스가 '{local_path}'에 저장되었습니다.")

In [ ]:
# 저장된 경로에서 인덱스를 다시 로드합니다.
new_vectorstore = FAISS.load_local(
    folder_path=local_path,
    embeddings=embeddings, # 임베딩 함수는 로드 시에도 필요합니다.
    allow_dangerous_deserialization=True # 최신 버전에서 보안을 위해 필요
)

print("✅ Faiss 인덱스가 로컬에서 메모리로 로드되었습니다.")

In [ ]:
texts_to_add = [
    "벡터 저장소는 인공지능 애플리케이션의 핵심 구성 요소입니다.",
    "Faiss는 L2 거리 및 내적을 포함한 다양한 거리 측정법을 지원합니다."
]

# 각 텍스트에 적용할 메타데이터 리스트를 전달합니다.
metadata_list = [{"source": "news"}, {"source": "tech_blog"}]

# 기존 vectorstore 객체에 새로운 텍스트 추가
new_ids = vectorstore.add_texts(texts_to_add, metadatas=metadata_list)

print("✅ 새로운 텍스트 추가 완료")
print(f"새로 추가된 ID (제거 시 사용): {new_ids}")

In [ ]:
query = "Faiss의 주요 목적은 무엇인가요?"

# 벡터 저장소에서 가장 유사한 상위 2개의 문서를 검색
results = vectorstore.similarity_search(query, k=2)

print(f"--- 🔎 검색 결과 (Top {len(results)}건) ---")
for i, doc in enumerate(results):
    print(f"[{i+1}] 내용: {doc.page_content[:40]}...")
    print(f"    출처: {doc.metadata.get('source')}")

In [ ]:
# 이전 add_texts() 호출에서 반환된 ID 중 첫 번째 ID를 예시로 사용
# ID 기반 문서 제거만 지원 --> 쿼리는 지원하지 않음
id_to_remove = new_ids[0] 

print(f"--- ⚠️ ID '{id_to_remove}'를 가진 문서를 제거합니다. ---")

# delete() 메서드를 사용하여 해당 ID에 해당하는 벡터와 문서를 제거합니다.
# Faiss는 인덱스를 재구성하여 해당 ID를 삭제합니다.
vectorstore.delete([id_to_remove])

print("✅ 문서 제거 완료")

# (확인) 제거 후, 해당 ID가 포함되었던 문서의 내용을 검색해 보면 다른 결과가 나올 수 있습니다.

## ChormaDB와 Faiss의 차이점

  ChromaDB = 벡터를 저장하고 관리하는 DB  
  FAISS = 벡터를 빠르게 검색하는 인덱스 라이브러리

  실습 관점에서는:

  - PDF 문서를 저장해두고 계속 검색/추가/삭제하려면 Chroma
  - 빠른 유사도 검색 실습이나 메모리 기반 검색을 해보려면 FAISS

두 기술 모두 텍스트를 숫자로 변환한 '벡터'를 다루지만, 태생과 목적이 조금 다릅니다.


1. FAISS (Facebook AI Similarity Search)
Meta(구 Facebook)에서 개발한 순수 벡터 검색 라이브러리입니다.

특징: 오로지 "수많은 숫자 배열(벡터) 중에서 가장 비슷한 것을 얼마나 빨리 찾아낼 것인가?"에 집중합니다.

장점: C++ 기반으로 작성되어 속도가 압도적으로 빠르며, 수백만~수십억 건의 대규모 데이터 검색에 최적화되어 있습니다. GPU 연산도 완벽하게 지원합니다.

단점: '데이터베이스'가 아니기 때문에 문서의 원본 텍스트나 메타데이터(예: 페이지 번호, 작성자 등)를 자체적으로 저장하지 않습니다. 벡터 값만 저장하므로, 검색된 벡터가 어떤 문서인지 매핑해 주는 작업(ID 관리 등)을 개발자가 직접 신경 써야 합니다. 영구 저장도 인덱스를 파일로 쓰고 읽는 방식을 취합니다.

2. Chroma DB
오직 AI와 LLM, 특히 RAG 애플리케이션 구축을 위해 만들어진 오픈소스 벡터 데이터베이스입니다.

특징: 개발자 친화적이며, 벡터 데이터뿐만 아니라 원본 문서, 메타데이터를 한 통에 담아 관리할 수 있습니다.

장점: 앞서 작성하셨던 코드처럼 from_documents() 함수 하나만 쓰면, 임베딩부터 텍스트 저장, 메타데이터 연동, 로컬 파일 저장(persist_directory)까지 알아서 척척 해냅니다. Langchain과의 궁합이 매우 좋고 초기 세팅이 매우 쉽습니다.

단점: FAISS처럼 극한의 인덱스 튜닝(HNSW, PQ 등)을 하거나 수백만 건 이상의 초대규모 데이터를 밀리초 단위로 검색하는 하드코어한 성능 측면에서는 FAISS보다 조금 무겁고 느릴 수 있습니다.